# Lab 02 – PDF, DOCX & API Data Processing
**Tasks 1–9** | Python (PyPDF2/pdfplumber, python-docx, pandas, requests)

**Before running:** set the file paths in the *Setup* cell below to your own files
(a PDF with 5+ pages, DOCX files, a DOCX with a table). Tasks 7–9 need an internet connection.

## Setup

In [3]:
!pip install pdfplumber PyPDF2 python-docx nltk pandas requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [4]:
# Uncomment if packages are missing:
# !pip install pdfplumber PyPDF2 python-docx nltk pandas requests

import re, csv, math, pprint, sys
from collections import Counter

import pdfplumber
import docx                     # python-docx
from docx.text.paragraph import Paragraph
import pandas as pd
import requests
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

# ====== CHANGE THESE PATHS TO YOUR OWN FILES ======
PDF_PATH        = "sample.pdf"        # PDF with at least 5 pages (Tasks 1, 2, 3)
DOCX_PATH       = "sample1.docx"       # any DOCX with headings/paragraphs (Task 4)
DOCX_TABLE_PATH = "sample_table.docx" # DOCX containing at least one table (Task 5)
DOCX_A_PATH     = "doc_a.docx"        # Document A (Task 6)
DOCX_B_PATH     = "doc_b.docx"        # Document B (Task 6)
# ==================================================

# Shared stopword list (30+ words) used in Tasks 1 and 6
COMMON_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "of", "to", "in", "on", "at", "for",
    "with", "by", "from", "is", "are", "was", "were", "be", "been", "it", "its",
    "this", "that", "these", "those", "as", "not", "have", "has", "had", "will",
    "can", "which", "their", "they", "we", "you", "i"
}

---
# Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

### (a) Extract text page by page

In [5]:
with pdfplumber.open(PDF_PATH) as pdf:
    pages_text = [(page.extract_text() or "") for page in pdf.pages]

print("Total pages extracted:", len(pages_text))

Total pages extracted: 11


### (b) Merge, clean with `re.sub()`, tokenize

In [6]:
full_text = " ".join(pages_text)

cleaned = re.sub(r"[^\w\s]", "", full_text)   # remove punctuation
cleaned = re.sub(r"[\d_]", "", cleaned)         # remove digits (and underscores)
tokens = cleaned.lower().split()                  # lowercase + split on whitespace

print("Total token count:", len(tokens))

Total token count: 8136


### (c) Word frequency (stopwords removed) – top 10

In [7]:
print("Stopwords used:", len(COMMON_STOPWORDS))

freq = {}
for word in tokens:
    if word not in COMMON_STOPWORDS:
        freq[word] = freq.get(word, 0) + 1

top10 = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("\nTop 10 most frequent words:")
for rank, (word, count) in enumerate(top10, 1):
    print(f"{rank:>2}. {word:<15} {count}")

Stopwords used: 40

Top 10 most frequent words:
 1. data            782
 2. lake            235
 3. warehouse       73
 4. big             41
 5. into            39
 6. processing      35
 7. stored          34
 8. warehouses      33
 9. metadata        33
10. lakes           31


### (d) Keyword search – sentences with page numbers

In [8]:
keyword = input("Enter a keyword to search: ").strip().lower()

found = False
for page_no, text in enumerate(pages_text, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
    for sentence in sentences:
        if keyword and keyword in sentence.lower():
            print(f"[Page {page_no}] {sentence.strip()}")
            found = True

if not found:
    print(f"No sentence containing '{keyword}' was found in the document.")

Enter a keyword to search: lake
[Page 1] ITM Web of Conferences 17, 03025 (2018) https://doi.org/10.1051/itmconf/20181703025 WCSN 2017 Data lake: a new ideology in big data era Pwint Phyu Khine1,a,b,* and Zhao Shun Wang1,c 1Department of Computer Science and Technology, School of Computer and Communication Engineering, University of Science and Technology Beijing (USTB), 10083, Beijing, China 13051513579, apwintphyukhinecs@gmail.com; bpwintphyukhinecs@outlook.com; czhswang@sohu.com Abstract.
[Page 1] Data Lake is one of the arguable concepts appeared in the era of big data.
[Page 1] Data Lake original idea is originated from business field instead of academic field.
[Page 1] As Data Lake is a newly conceived idea with revolutionized concepts, it brings many challenges for its adoption.
[Page 1] However, the potential to change the data landscape makes the research of Data Lake worthwhile.
[Page 1] 1 Introduction In the era of big data, a new term called “Data Lake” came into view of th

---
# Task 2: Structured Page Metadata Extraction and CSV Export

### (a) Build `pages_data`

In [9]:
pages_data = []
with pdfplumber.open(PDF_PATH) as pdf:
    if len(pdf.pages) < 5:
        print("Warning: PDF has fewer than 5 pages (task requires at least 5).")
    for i, page in enumerate(pdf.pages, start=1):
        raw = page.extract_text() or ""
        pages_data.append({
            "page_number": i,
            "raw_text": raw,
            "word_count": len(raw.split()),
            "char_count": len(re.sub(r"\s", "", raw)),   # excluding spaces
        })

print("First 2 entries:")
for entry in pages_data[:2]:
    pprint.pprint(entry)
    print()

First 2 entries:
{'char_count': 2958,
 'page_number': 1,
 'raw_text': 'ITM Web of Conferences 17, 03025 (2018) '
             'https://doi.org/10.1051/itmconf/20181703025\n'
             'WCSN 2017\n'
             'Data lake: a new ideology in big data era\n'
             'Pwint Phyu Khine1,a,b,* and Zhao Shun Wang1,c\n'
             '1Department of Computer Science and Technology, School of '
             'Computer and Communication\n'
             'Engineering, University of Science and Technology Beijing '
             '(USTB), 10083, Beijing, China\n'
             '13051513579, apwintphyukhinecs@gmail.com; '
             'bpwintphyukhinecs@outlook.com;\n'
             'czhswang@sohu.com\n'
             'Abstract. Data Lake is one of the arguable concepts appeared\n'
             'in the era of big data. Data Lake original idea is originated '
             'from\n'
             'business field instead of academic field. As Data Lake is a\n'
             'newly conceived idea with re

### (b) Highest / lowest word count and average

In [10]:
highest = max(pages_data, key=lambda d: d["word_count"])
print(f"(i) Highest word count: page {highest['page_number']} ({highest['word_count']} words)")
print("Full content:\n", highest["raw_text"])

non_zero = [d for d in pages_data if d["word_count"] > 0]
if non_zero:
    lowest = min(non_zero, key=lambda d: d["word_count"])
    print(f"\n(ii) Lowest non-zero word count: page {lowest['page_number']} ({lowest['word_count']} words)")
else:
    print("\n(ii) All pages returned zero words.")

empty_pages = [d["page_number"] for d in pages_data if d["word_count"] == 0]
if empty_pages:
    print("     Pages with zero words (skipped for lowest):", empty_pages)

avg = round(sum(d["word_count"] for d in pages_data) / len(pages_data), 2) if pages_data else 0
print(f"\n(iii) Average word count across all pages: {avg}")

(i) Highest word count: page 9 (1250 words)
Full content:
 ITM Web of Conferences 17, 03025 (2018) https://doi.org/10.1051/itmconf/20181703025
WCSN 2017
4 Concerns and challenges of data lake 5 Discussion and conclusion
Data Lakes try to solve two problems – data silos (old problem) and challenges imposed by
4.1 Data lake concerns big data initiatives (new problem) [6]. Instead of having independently data collections, all
data to be stored are collected in Data Lake to handle the old silos problem. The new
There are two primary concerns for Data Lake. From business point of view, Data Lake
problem is handling the challenges of big data era i.e. data lakes try to solve the challenges
may be another marketing hype for Hadoop. From Technical point of view, Data Lake can
imposed by big data V’s characteristics – volume, velocity, verity, variety and value.
easily be changed into data swamps [18].
If data generated or produced from different departments within an organization are
Marketing

### (c) Case-insensitive phrase search with custom exception

In [11]:
class PhraseNotFoundError(Exception):
    """Raised when the search phrase is not found on any page."""
    pass

phrase = input("Enter a search phrase: ").strip()

try:
    matches = []
    for d in pages_data:
        count = d["raw_text"].lower().count(phrase.lower()) if phrase else 0
        if count > 0:
            matches.append((d["page_number"], count))

    if not matches:
        raise PhraseNotFoundError(f"The phrase '{phrase}' was not found in any page.")

    for page_no, count in matches:
        print(f"Page {page_no}: '{phrase}' appears {count} time(s)")
except PhraseNotFoundError as e:
    print("PhraseNotFoundError caught:", e)

Enter a search phrase: Data Lake
Page 1: 'Data Lake' appears 19 time(s)
Page 2: 'Data Lake' appears 13 time(s)
Page 3: 'Data Lake' appears 17 time(s)
Page 4: 'Data Lake' appears 11 time(s)
Page 5: 'Data Lake' appears 28 time(s)
Page 6: 'Data Lake' appears 14 time(s)
Page 7: 'Data Lake' appears 29 time(s)
Page 8: 'Data Lake' appears 25 time(s)
Page 9: 'Data Lake' appears 41 time(s)
Page 10: 'Data Lake' appears 18 time(s)
Page 11: 'Data Lake' appears 24 time(s)


### (d) Write and re-read `pdf_summary.csv` (csv module)

In [12]:
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for d in pages_data:
        writer.writerow([d["page_number"], d["word_count"], d["char_count"]])

print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

Contents of pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '528', '2958']
['2', '584', '3234']
['3', '1146', '6267']
['4', '502', '2558']
['5', '1103', '5818']
['6', '467', '2623']
['7', '1081', '5917']
['8', '608', '3144']
['9', '1250', '6537']
['10', '493', '3103']
['11', '708', '4718']


---
# Task 3: PDF Text Preprocessing Pipeline for NLP

### (a) Extract all text and clean with `re.sub()`

In [13]:
with pdfplumber.open(PDF_PATH) as pdf:
    raw_text = " ".join((p.extract_text() or "") for p in pdf.pages)

before_len = len(raw_text)

text = re.sub(r"https?://\S+|www\.\S+", "", raw_text)   # (i)   remove URLs
text = re.sub(r"\d", "", text)                            # (ii)  remove digits
text = re.sub(r"[^A-Za-z\s.]", "", text)                  # (iii) keep letters, spaces, full stops
text = re.sub(r" +", " ", text)                           # (iv)  collapse multiple spaces
# note: newlines are also treated as whitespace below
text = re.sub(r"\s*\n\s*", " ", text)
text = re.sub(r" +", " ", text)

print("Character count BEFORE cleaning:", before_len)
print("Character count AFTER cleaning: ", len(text))

Character count BEFORE cleaning: 55346
Character count AFTER cleaning:  51749


### (b) Lowercase and split into sentences

In [14]:
text = text.lower()
sentences_raw = re.split(r"[.!?]", text)
print("Sentence count BEFORE removing empty strings:", len(sentences_raw))

sentences = [s.strip() for s in sentences_raw]
sentences = [s for s in sentences if s]
print("Sentence count AFTER removing empty strings: ", len(sentences))

Sentence count BEFORE removing empty strings: 560
Sentence count AFTER removing empty strings:  540


### (c) Tokenize, remove NLTK stopwords, filter short sentences

In [15]:
stop_words = set(stopwords.words("english"))

filtered = []          # list of (original_sentence, tokens_without_stopwords)
for s in sentences:
    words = re.findall(r"[a-z]+", s)
    kept = [w for w in words if w not in stop_words]
    filtered.append((s, kept))

print("Sentence count BEFORE token filter:", len(filtered))

final_sentences = [s for s, kept in filtered if len(kept) >= 5]
print("Sentence count AFTER token filter: ", len(final_sentences))

print("\n3 sample sentences:")
for s in final_sentences[:3]:
    print("-", s)

Sentence count BEFORE token filter: 540
Sentence count AFTER token filter:  411

3 sample sentences:
- itm web of conferences wcsn data lake a new ideology in big data era pwint phyu khineab and zhao shun wangc department of computer science and technology school of computer and communication engineering university of science and technology beijing ustb beijing china apwintphyukhinecsgmail
- data lake is one of the arguable concepts appeared in the era of big data
- data lake original idea is originated from business field instead of academic field


### (d) Write to `nlp_ready_output.txt` and verify

In [16]:
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in final_sentences:
        f.write(s + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

print("(i)   Total lines written:", len(lines))
if lines:
    print("(ii)  Longest sentence :", max(lines, key=len))
    print("(iii) Shortest sentence:", min(lines, key=len))

(i)   Total lines written: 411
(ii)  Longest sentence : comparison of data warehouse and data lake comparison data warehouse data lake data structured structuredsemistructured processed data unstructured data raw data unprocessed data processing schemaonwrite schemaonread storage expensive reliable low cost storage agility less agile fixed high agility flexible configuration configuration security matured maturing users business data scientists especially professional those familiar with domain
(iii) Shortest sentence: data flow should go along olap


---
# Task 4: DOCX Paragraph Classification and Statistical Analysis

### (a) Read paragraphs with text and style

In [17]:
doc = docx.Document(DOCX_PATH)

para_list = [{"text": p.text, "style": p.style.name} for p in doc.paragraphs]

print("Total paragraph count:", len(para_list))
print("Unique style names:", sorted({p["style"] for p in para_list}))

Total paragraph count: 46
Unique style names: ['Normal']


### (b) Classify into Headings / Body / Other-Empty

In [18]:
headings, body, other = [], [], []

for p in para_list:
    style = p["style"]
    if not p["text"].strip():                       # blank paragraphs -> Other/Empty
        other.append(p)
    elif style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings     :", len(headings))
print("Body         :", len(body))
print("Other / Empty:", len(other))
print("Total        :", len(headings) + len(body) + len(other))

Headings     : 0
Body         : 45
Other / Empty: 1
Total        : 46


### (c) Statistics for body paragraphs (> 5 words)

In [19]:
qualifying = []
for p in body:
    wc = len(p["text"].split())
    if wc > 5:
        qualifying.append({
            "text": p["text"],
            "word_count": wc,
            "char_count": len(p["text"].replace(" ", "")),   # excluding spaces
        })

if qualifying:
    highest = max(qualifying, key=lambda d: d["word_count"])
    lowest  = min(qualifying, key=lambda d: d["word_count"])
    mean_wc = round(sum(d["word_count"] for d in qualifying) / len(qualifying), 2)

    print(f"(i)   Highest word count ({highest['word_count']} words, {highest['char_count']} chars):\n{highest['text']}\n")
    print(f"(ii)  Lowest word count above 5 ({lowest['word_count']} words, {lowest['char_count']} chars):\n{lowest['text']}\n")
    print(f"(iii) Mean word count of qualifying body paragraphs: {mean_wc}")
else:
    print("No body paragraphs with more than 5 words were found.")

(i)   Highest word count (57 words, 299 chars):
(a) Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.
https://jsonplaceholder.typicode.com/todos

(ii)  Lowest word count above 5 (8 words, 40 chars):
Task 3: PDF Text Preprocessing Pipeline for NLP

(iii) Mean word count of qualifying body paragraphs: 35.93


### (d) Nested dictionary by style (sorted by count, max 2 texts)

In [20]:
style_map = {}
for p in para_list:
    entry = style_map.setdefault(p["style"], {"count": 0, "texts": []})
    entry["count"] += 1
    entry["texts"].append(p["text"])

# Sort outer dict by count (descending)
style_map = dict(sorted(style_map.items(), key=lambda kv: kv[1]["count"], reverse=True))

# If a style has more than 2 paragraphs, show only the first 2 texts
display_map = {
    style: {"count": info["count"], "texts": info["texts"][:2] if info["count"] > 2 else info["texts"]}
    for style, info in style_map.items()
}

pprint.pprint(display_map, width=100)

{'Normal': {'count': 46,
            'texts': ['Task 1: Multi-Page PDF Extraction with Word Frequency Analysis',
                      '(a) Open a PDF file using PyPDF2 or pdfplumber and extract text from each '
                      'page individually. Store the result as a list of strings where each index '
                      'represents one page. Print the total number of pages extracted.']}}


---
# Task 5: DOCX Table Extraction and Pandas Integration

### (a) Extract all tables as lists of lists

In [21]:
doc5 = docx.Document(DOCX_TABLE_PATH)

all_tables = []
for table in doc5.tables:
    all_tables.append([[cell.text.strip() for cell in row.cells] for row in table.rows])

print("Total number of tables:", len(all_tables))
for i, t in enumerate(all_tables, 1):
    rows = len(t)
    cols = len(t[0]) if t else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

Total number of tables: 1
Table 1: 5 rows x 3 columns


### (b) First table -> DataFrame, count empty cells

In [22]:
first = all_tables[0]
df = pd.DataFrame(first[1:], columns=first[0])
print(df.to_string())

empty_counts = df.apply(lambda col: col.apply(lambda v: v is None or str(v).strip() == "").sum())
print("\nEmpty / None cells per column:")
print(empty_counts)

                     Normal Form         Primary Requirement                                                                                             Description
0        1NF (First Normal Form)               Atomic Values                   Every attribute contains only atomic (indivisible) values, and each record is unique.
1       2NF (Second Normal Form)     No Partial Dependencies  Must be in 1NF. All non-key attributes must be fully functionally dependent on the entire primary key.
2        3NF (Third Normal Form)  No Transitive Dependencies                      Must be in 2NF. No non-key attribute is transitively dependent on the primary key.
3  BCNF (Boyce-Codd Normal Form)       Superkey Determinants                               For every non-trivial functional dependency X -> Y, X must be a superkey.

Empty / None cells per column:
Normal Form            0
Primary Requirement    0
Description            0
dtype: int64


### (c) Paragraphs before the first table

In [23]:
pre_table_paras = []
for child in doc5.element.body.iterchildren():
    if child.tag.endswith("}tbl"):        # w:tbl -> stop
        break
    if child.tag.endswith("}p"):          # w:p -> paragraph
        pre_table_paras.append(Paragraph(child, doc5))

print("Paragraphs before the first table:", len(pre_table_paras))

Paragraphs before the first table: 3


### (d) Write `docx_extracted.txt`

In [24]:
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table_paras:
        f.write(p.text + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first:
        f.write(" | ".join(row) + "\n")

print(open("docx_extracted.txt", encoding="utf-8").read())

--- PRE-TABLE PARAGRAPHS ---
Database Systems: Normalization & Schema Design
This document serves as a study reference for relational database schema design, Entity-Relationship Diagrams (ERDs), and functional dependency normalization. Proper normalization up to Boyce-Codd Normal Form (BCNF) minimizes redundancy and protects data integrity.
Summary of Normal Forms

--- TABLE DATA ---
Normal Form | Primary Requirement | Description
1NF (First Normal Form) | Atomic Values | Every attribute contains only atomic (indivisible) values, and each record is unique.
2NF (Second Normal Form) | No Partial Dependencies | Must be in 1NF. All non-key attributes must be fully functionally dependent on the entire primary key.
3NF (Third Normal Form) | No Transitive Dependencies | Must be in 2NF. No non-key attribute is transitively dependent on the primary key.
BCNF (Boyce-Codd Normal Form) | Superkey Determinants | For every non-trivial functional dependency X -> Y, X must be a superkey.



---
# Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

### (a) Compare headings using set operations

In [25]:
doc_a = docx.Document(DOCX_A_PATH)
doc_b = docx.Document(DOCX_B_PATH)

def get_headings(d):
    return [p.text.strip() for p in d.paragraphs
            if p.style.name.startswith("Heading") and p.text.strip()]

headings_a = get_headings(doc_a)
headings_b = get_headings(doc_b)
set_a, set_b = set(headings_a), set(headings_b)

print("(i)   Headings common to both:")
for h in sorted(set_a & set_b): print("   -", h)
print("(ii)  Headings only in Document A:")
for h in sorted(set_a - set_b): print("   -", h)
print("(iii) Headings only in Document B:")
for h in sorted(set_b - set_a): print("   -", h)

(i)   Headings common to both:
(ii)  Headings only in Document A:
(iii) Headings only in Document B:


### (b) Merge "Normal" paragraphs and deduplicate

In [26]:
def get_normal(d):
    # blank paragraphs are ignored
    return [p.text for p in d.paragraphs if p.style.name == "Normal" and p.text.strip()]

merged = get_normal(doc_a) + get_normal(doc_b)
print("Total paragraphs BEFORE deduplication:", len(merged))

seen, unique_paras = set(), []
for para in merged:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        unique_paras.append(para)

print("Total paragraphs AFTER deduplication: ", len(unique_paras))

Total paragraphs BEFORE deduplication: 90
Total paragraphs AFTER deduplication:  45


### (c) Combined word frequency (Counter)

In [27]:
word_counter = Counter()
for para in unique_paras:
    clean = re.sub(r"[^\w\s]", "", para.lower())
    clean = re.sub(r"[\d_]", "", clean)
    words = [w for w in clean.split() if w not in COMMON_STOPWORDS]
    word_counter.update(words)

print("Top 20 most frequent words:")
for rank, (word, count) in enumerate(word_counter.most_common(20), 1):
    print(f"{rank:>2}. {word:<15} {count}")

Top 20 most frequent words:
 1. print           36
 2. count           23
 3. all             22
 4. each            19
 5. list            18
 6. total           14
 7. word            13
 8. into            13
 9. ii              13
10. paragraphs      13
11. using           11
12. page            11
13. b               11
14. iii             11
15. file            10
16. users           10
17. task            9
18. remove          9
19. c               9
20. dictionary      9


### (d) Summary report

In [28]:
para_word_counts = [len(p.split()) for p in unique_paras]
avg_len = round(sum(para_word_counts) / len(para_word_counts), 2) if para_word_counts else 0
longest = max(unique_paras, key=lambda p: len(p.split())) if unique_paras else ""

print("=" * 50)
print("SUMMARY REPORT")
print("=" * 50)
print("(i)   Total headings across both documents :", len(headings_a) + len(headings_b))
print("(ii)  Total unique body paragraphs          :", len(unique_paras))
print("(iii) Vocabulary size (unique word types)   :", len(word_counter))
print("(iv)  Average paragraph length (words)      :", avg_len)
print("(v)   Longest paragraph (%d words):" % (len(longest.split())))
print(longest)

SUMMARY REPORT
(i)   Total headings across both documents : 0
(ii)  Total unique body paragraphs          : 45
(iii) Vocabulary size (unique word types)   : 422
(iv)  Average paragraph length (words)      : 35.93
(v)   Longest paragraph (57 words):
(a) Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.
https://jsonplaceholder.typicode.com/todos


---
# Task 7: Multi-Endpoint API Data Joining and Analysis

### (a) Fetch users and posts with status validation

In [29]:
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

def fetch_json(url):
    resp = requests.get(url, timeout=15)
    if resp.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {resp.status_code}")
    return resp.json()

users = fetch_json(USERS_URL)
posts = fetch_json(POSTS_URL)
print(f"Fetched {len(users)} users and {len(posts)} posts.")

Fetched 10 users and 100 posts.


### (b) Join users and posts

In [30]:
joined = {}
for u in users:
    user_posts = [p["title"] for p in posts if p["userId"] == u["id"]]
    joined[u["name"]] = {
        "email": u["email"],
        "post_count": len(user_posts),
        "titles": user_posts,
    }

for name in list(joined)[:3]:
    print(name)
    pprint.pprint(joined[name], width=110)
    print()

Leanne Graham
{'email': 'Sincere@april.biz',
 'post_count': 10,
 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
            'qui est esse',
            'ea molestias quasi exercitationem repellat qui ipsa sit aut',
            'eum et est occaecati',
            'nesciunt quas odio',
            'dolorem eum magni eos aperiam quia',
            'magnam facilis autem',
            'dolorem dolore est ipsam',
            'nesciunt iure omnis dolorem tempora et accusantium',
            'optio molestias id quia eum']}

Ervin Howell
{'email': 'Shanna@melissa.tv',
 'post_count': 10,
 'titles': ['et ea vero quia laudantium autem',
            'in quibusdam tempore odit est dolorem',
            'dolorum ut in voluptas mollitia et saepe quo animi',
            'voluptatem eligendi optio',
            'eveniet quod temporibus',
            'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio',
            'fugit voluptas sed molestias v

### (c) User with the most posts

In [31]:
top_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = next(u for u in users if u["name"] == top_name)

print("(i)   Full name   :", top_name)
print("(ii)  Email       :", joined[top_name]["email"])
print("(iii) Company name:", top_user["company"]["name"])
print("(iv)  Post titles :")
for i, title in enumerate(joined[top_name]["titles"], 1):
    print(f"   {i}. {title}")

(i)   Full name   : Leanne Graham
(ii)  Email       : Sincere@april.biz
(iii) Company name: Romaguera-Crona
(iv)  Post titles :
   1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
   2. qui est esse
   3. ea molestias quasi exercitationem repellat qui ipsa sit aut
   4. eum et est occaecati
   5. nesciunt quas odio
   6. dolorem eum magni eos aperiam quia
   7. magnam facilis autem
   8. dolorem dolore est ipsam
   9. nesciunt iure omnis dolorem tempora et accusantium
   10. optio molestias id quia eum


### (d) Filter users by company.bs / city length

In [32]:
print(f"{'Name':<25}{'City':<18}{'Posts'}")
print("-" * 50)
for u in users:
    has_synergies = "synergies" in u["company"]["bs"].lower()
    long_city = len(u["address"]["city"]) > 8
    if has_synergies or long_city:
        print(f"{u['name']:<25}{u['address']['city']:<18}{joined[u['name']]['post_count']}")

Name                     City              Posts
--------------------------------------------------
Leanne Graham            Gwenborough       10
Ervin Howell             Wisokyburgh       10
Clementine Bauch         McKenziehaven     10
Patricia Lebsack         South Elvis       10
Chelsey Dietrich         Roscoeview        10
Mrs. Dennis Schulist     South Christy     10
Kurtis Weissnat          Howemouth         10
Nicholas Runolfsdottir V Aliyaview         10
Glenna Reichert          Bartholomebury    10
Clementina DuBuque       Lebsackbury       10


---
# Task 8: To-Do Completion Rate Analysis Across Users

### (a) Fetch todos with validation

In [33]:
TODOS_URL = "https://jsonplaceholder.typicode.com/todos"

resp = requests.get(TODOS_URL, timeout=15)
if resp.status_code != 200:
    print(f"Error: expected status 200 but got {resp.status_code}")
    raise SystemExit
todos = resp.json()
if not isinstance(todos, list):
    print("Error: parsed response is not a list.")
    raise SystemExit
if len(todos) < 1:
    print("Error: the list contains no records.")
    raise SystemExit

print("Total todo records retrieved:", len(todos))

Total todo records retrieved: 200


### (b) Group by userId

In [34]:
stats = {}
for t in todos:
    s = stats.setdefault(t["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    s["total"] += 1
    if t["completed"]:
        s["completed"] += 1
    else:
        s["incomplete"] += 1

for uid in sorted(stats):
    s = stats[uid]
    print(f"User {uid}: {s}  ->  completed + incomplete == total: {s['completed'] + s['incomplete'] == s['total']}")

User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  completed + incomplete == total: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  completed + incomplete == total: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  ->  completed + incomplete == total: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  completed + incomplete == total: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  completed + incomplete == total: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  completed + incomplete == total: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  ->  completed + incomplete == total: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  completed + incomplete == total: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  completed + incomplete == total: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  completed + incomplete == total: True


### (c) Completion rates

In [35]:
rates = {uid: round(s["completed"] / s["total"] * 100, 1) for uid, s in stats.items()}

best  = max(rates, key=rates.get)
worst = min(rates, key=rates.get)
overall = round(sum(rates.values()) / len(rates), 1)

print(f"(i)   Highest completion rate: userId {best} -> {rates[best]}%")
print(f"(ii)  Lowest completion rate : userId {worst} -> {rates[worst]}%")
print(f"(iii) Overall average completion rate: {overall}%")

(i)   Highest completion rate: userId 5 -> 60.0%
(ii)  Lowest completion rate : userId 4 -> 30.0%
(iii) Overall average completion rate: 45.0%


### (d) Formatted ranking table

In [36]:
users8 = requests.get("https://jsonplaceholder.typicode.com/users", timeout=15).json()
name_lookup = {u["id"]: u["name"] for u in users8}

ranked = sorted(rates.items(), key=lambda kv: kv[1], reverse=True)

header = f"{'Rank':<6}| {'Name':<25}| {'Total Tasks':<12}| {'Completed':<10}| {'Completion Rate %'}"
print(header)
print("-" * len(header))
for rank, (uid, rate) in enumerate(ranked, 1):
    print(f"{rank:<6}| {name_lookup[uid]:<25}| {stats[uid]['total']:<12}| {stats[uid]['completed']:<10}| {rate}")

Rank  | Name                     | Total Tasks | Completed | Completion Rate %
------------------------------------------------------------------------------
1     | Chelsey Dietrich         | 20          | 12        | 60.0
2     | Clementina DuBuque       | 20          | 12        | 60.0
3     | Leanne Graham            | 20          | 11        | 55.0
4     | Nicholas Runolfsdottir V | 20          | 11        | 55.0
5     | Kurtis Weissnat          | 20          | 9         | 45.0
6     | Ervin Howell             | 20          | 8         | 40.0
7     | Glenna Reichert          | 20          | 8         | 40.0
8     | Clementine Bauch         | 20          | 7         | 35.0
9     | Patricia Lebsack         | 20          | 6         | 30.0
10    | Mrs. Dennis Schulist     | 20          | 6         | 30.0


---
# Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

### (a) Flatten user records

In [37]:
users9 = requests.get("https://jsonplaceholder.typicode.com/users", timeout=15).json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

flat_users = [flatten_user(u) for u in users9]
pprint.pprint(flat_users, width=120)

[{'city': 'Gwenborough',
  'company_catchphrase': 'Multi-layered client-server neural-net',
  'company_name': 'Romaguera-Crona',
  'email': 'Sincere@april.biz',
  'id': 1,
  'lat': '-37.3159',
  'lng': '81.1496',
  'name': 'Leanne Graham',
  'username': 'Bret',
  'zipcode': '92998-3874'},
 {'city': 'Wisokyburgh',
  'company_catchphrase': 'Proactive didactic contingency',
  'company_name': 'Deckow-Crist',
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'lat': '-43.9509',
  'lng': '-34.4618',
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'zipcode': '90566-7771'},
 {'city': 'McKenziehaven',
  'company_catchphrase': 'Face to face bifurcated interface',
  'company_name': 'Romaguera-Jacobson',
  'email': 'Nathan@yesenia.net',
  'id': 3,
  'lat': '-68.6102',
  'lng': '-47.0653',
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'zipcode': '59590-4157'},
 {'city': 'South Elvis',
  'company_catchphrase': 'Multi-tiered zero tolerance productivity',
  'company_name': 'Robel-Corkery'

### (b) DataFrame, float casting, distance from origin

In [38]:
df9 = pd.DataFrame(flat_users)
print(df9.to_string())

df9["lat"] = df9["lat"].astype(float)
df9["lng"] = df9["lng"].astype(float)
df9["distance_from_origin"] = df9.apply(lambda r: math.sqrt(r["lat"] ** 2 + r["lng"] ** 2), axis=1)

far = df9.loc[df9["distance_from_origin"].idxmax()]
print(f"\nUser with greatest distance from origin: {far['name']} ({far['distance_from_origin']:.2f})")

   id                      name          username                      email            city     zipcode       lat        lng        company_name                       company_catchphrase
0   1             Leanne Graham              Bret          Sincere@april.biz     Gwenborough  92998-3874  -37.3159    81.1496     Romaguera-Crona    Multi-layered client-server neural-net
1   2              Ervin Howell         Antonette          Shanna@melissa.tv     Wisokyburgh  90566-7771  -43.9509   -34.4618        Deckow-Crist            Proactive didactic contingency
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   McKenziehaven  59590-4157  -68.6102   -47.0653  Romaguera-Jacobson         Face to face bifurcated interface
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org     South Elvis  53919-4257   29.4572  -164.2990       Robel-Corkery  Multi-tiered zero tolerance productivity
4   5          Chelsey Dietrich            Kamren   Lucio_He

### (c) Hemisphere classification

In [39]:
north = df9[df9["lat"] > 0]
south = df9[df9["lat"] < 0]

print("Northern Hemisphere (lat > 0):", list(north["name"]))
print("Southern Hemisphere (lat < 0):", list(south["name"]))

avg_n = north["distance_from_origin"].mean() if len(north) else float("nan")
avg_s = south["distance_from_origin"].mean() if len(south) else float("nan")
print(f"\nAverage distance - North: {avg_n:.2f} | South: {avg_s:.2f}")
print("Higher average distance:", "Northern Hemisphere" if avg_n > avg_s else "Southern Hemisphere")

Northern Hemisphere (lat > 0): ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere (lat < 0): ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Average distance - North: 123.58 | South: 84.32
Higher average distance: Northern Hemisphere


### (d) Comment count per post

In [40]:
posts9    = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=15).json()
comments9 = requests.get("https://jsonplaceholder.typicode.com/comments", timeout=15).json()

comment_counts = Counter(c["postId"] for c in comments9)

post_stats = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts9]
post_stats.sort(key=lambda t: (-t[2], t[0]))     # descending by comment count

print("Top 5 most-commented posts:")
for t in post_stats[:5]:
    print("  ", t)

print("\nBottom 5 least-commented posts:")
for t in post_stats[-5:]:
    print("  ", t)

Top 5 most-commented posts:
   (1, 'sunt aut facere repellat provident occae', 5)
   (2, 'qui est esse', 5)
   (3, 'ea molestias quasi exercitationem repell', 5)
   (4, 'eum et est occaecati', 5)
   (5, 'nesciunt quas odio', 5)

Bottom 5 least-commented posts:
   (96, 'quaerat velit veniam amet cupiditate aut', 5)
   (97, 'quas fugiat ut perspiciatis vero provide', 5)
   (98, 'laboriosam dolor voluptates', 5)
   (99, 'temporibus sit alias delectus eligendi p', 5)
   (100, 'at nam consequatur ea labore ea harum', 5)
